# My Cross Track Collaboration project as a Data Engineer

#Nkwen Traders Data Cleaning

## Objective

##### The objective of this notebook is to clean the raw Nkwen Traders sales dataset and prepare a structured product catalogue in JSON format for use by the website's Catalog page.

### The cleaning process includes:

##### - Inspecting the raw dataset
##### - Handling missing values
##### - Standardizing inconsistent categories and payment methods
##### - Checking for duplicate transactions
##### - Validating numerical values
##### - Preparing the product catalogue
##### - Exporting the final data as products.json

In [1]:
# Importing libraries
import pandas as pd
import json

 # Loading the CSV file
df = pd.read_csv("Nkwen_traders_sales.csv")
df.head(10)


,TransactionID,Date,Product,Category,Quantity,UnitPrice_FCFA,TotalSale_FCFA,PaymentMethod,SalesRep,CustomerType
0,NKW-0001,2026-02-07,Rice 50kg,Grains,1.0,34928.0,34928.0,Bank Transfer,Divine K.,Walk-in
1,NKW-0002,2026-04-13,Onions 1kg,Produce,12.0,685.0,8220.0,Mobile Money,Florence A.,Wholesale
2,NKW-0003,2026-04-11,Salt 1kg,Groceries,11.0,297.0,3267.0,Orange Money,Beatrice T.,Walk-in
3,NKW-0004,2026-01-01,Bread (loaf),Bakery,13.0,591.0,7683.0,Bank Transfer,Divine K.,Wholesale
4,NKW-0005,2026-02-14,Beans (White),Grain,6.0,915.0,5490.0,Mobile Money,Beatrice T.,Walk-in
5,NKW-0006,2026-03-09,Rice 25kg,Grains,NaN,17880.0,35760.0,Bank Transfer,Florence A.,Walk-in
6,NKW-0007,2026-01-14,Bread (loaf),Bakery,19.0,614.0,11666.0,Orange Money,Florence A.,Walk-in
7,NKW-0008,2026-03-11,Rice 50kg,Grains,4.0,37414.0,149656.0,Bank Transfer,Ernest M.,Wholesale
8,NKW-0009,2026-06-26,Palm Oil 1L,Oils,15.0,1394.0,20910.0,Cash,Divine K.,Wholesale
9,NKW-0010,2026-01-05,Tomatoes 1kg,Produce,8.0,781.0,6248.0,Cash,Florence A.,Walk-in


##### From the sample data displayed above, the CSV is a sales transaction dataset, not a ready-made product catalogue. Therefore, for products.json, I’ld need to create one record per unique product rather than simply exporting all 500 transactions.

In [2]:
# Inspecting the data set
df.shape     
# Tells the number of columns and rows in the from the data set

(500, 10)

In [3]:
df.info() 
# Shows the data types and the number of Missing value counts per column

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   TransactionID   500 non-null    object 
 1   Date            500 non-null    object 
 2   Product         500 non-null    object 
 3   Category        500 non-null    object 
 4   Quantity        492 non-null    float64
 5   UnitPrice_FCFA  495 non-null    float64
 6   TotalSale_FCFA  500 non-null    float64
 7   PaymentMethod   490 non-null    object 
 8   SalesRep        494 non-null    object 
 9   CustomerType    500 non-null    object 
dtypes: float64(3), object(7)
memory usage: 39.2+ KB


In [4]:
# Checking the missing values
print(df.isnull().sum())

print()
print("Total number of missing values = ",df.isnull().sum().sum())

TransactionID      0
Date               0
Product            0
Category           0
Quantity           8
UnitPrice_FCFA     5
TotalSale_FCFA     0
PaymentMethod     10
SalesRep           6
CustomerType       0
dtype: int64

Total number of missing values =  29


In [5]:
# Checking for duplicate transactions
df.duplicated().sum()

0

### Duplicate Check
##### No duplicate transaction rows or duplicate TransactionID values were found in the raw dataset, so no records were removed for duplication.

In [6]:
# Check for Categories and its counts
df["Category"].value_counts()

Category
Produce       95
Groceries     82
Grains        80
Oils          74
Household     50
Groceres      30
Dairy         27
Grain         25
Bakery        22
House hold    15
Name: count, dtype: int64

##### From above, it shows that there are some inconsistencies. For example;
##### - Groceries and Groceres
##### It should be Groceries
##### - Also Grains and Grain
##### Should be Grains
##### - More so, Household and House hold
##### Should be Household 

In [7]:
# Fixing the categories
category_mapping = {
    "Groceres": "Groceries",
    "Grain": "Grains",
    "House hold": "Household"
}
df["Category"] = df["Category"].replace(category_mapping)
df["Category"].value_counts()

Category
Groceries    112
Grains       105
Produce       95
Oils          74
Household     65
Dairy         27
Bakery        22
Name: count, dtype: int64

In [8]:
# Checking payment method
df["PaymentMethod"].value_counts(dropna=False)

PaymentMethod
Bank Transfer    139
Mobile Money     131
Orange Money     114
Cash             105
NaN               10
Mobile Moeny       1
Name: count, dtype: int64

In [9]:
# From above, Mobile Moeny should be Mobile Money
df["PaymentMethod"] = df["PaymentMethod"].replace({"Mobile Moeny": "Mobile Money"})
df["PaymentMethod"].value_counts(dropna=False)

PaymentMethod
Bank Transfer    139
Mobile Money     132
Orange Money     114
Cash             105
NaN               10
Name: count, dtype: int64

In [11]:
# Dealing with missing values

In [10]:
df[df["Quantity"].isnull()]

,TransactionID,Date,Product,Category,Quantity,UnitPrice_FCFA,TotalSale_FCFA,PaymentMethod,SalesRep,CustomerType
5,NKW-0006,2026-03-09,Rice 25kg,Grains,NaN,17880.0,35760.0,Bank Transfer,Florence A.,Walk-in
25,NKW-0026,2026-05-21,Maggi Cubes (pack),Groceries,NaN,483.0,9177.0,Bank Transfer,Divine K.,Walk-in
29,NKW-0030,2026-02-12,Cassava (bag),Produce,NaN,3950.0,63200.0,Mobile Money,Divine K.,Wholesale
187,NKW-0188,2026-04-08,Onions 1kg,Produce,NaN,666.0,10656.0,Orange Money,Divine K.,Wholesale
189,NKW-0190,2026-05-22,Sugar 1kg,Groceries,NaN,728.0,2184.0,Bank Transfer,Ernest M.,Wholesale
353,NKW-0354,2026-04-17,Plantain (bunch),Produce,NaN,2603.0,36442.0,NaN,Florence A.,Walk-in
392,NKW-0393,2026-04-08,Tomato Paste (tin),Groceries,NaN,377.0,4901.0,Mobile Money,Ernest M.,Walk-in
444,NKW-0445,2026-06-12,Detergent 1kg,Household,NaN,1804.0,5412.0,Orange Money,Divine K.,Wholesale


##### From the above, the category Quantity has 8 missing values

In [12]:
# Calculating the missing Quantity values by dividing its total sale by its unit price
df["Quantity_Calculated"] = (
    df["TotalSale_FCFA"] / df["UnitPrice_FCFA"]
)
df[df["Quantity"].isnull()][
    ["Product", "Quantity", "UnitPrice_FCFA", "TotalSale_FCFA", "Quantity_Calculated"]
]


,Product,Quantity,UnitPrice_FCFA,TotalSale_FCFA,Quantity_Calculated
5,Rice 25kg,NaN,17880.0,35760.0,2.0
25,Maggi Cubes (pack),NaN,483.0,9177.0,19.0
29,Cassava (bag),NaN,3950.0,63200.0,16.0
187,Onions 1kg,NaN,666.0,10656.0,16.0
189,Sugar 1kg,NaN,728.0,2184.0,3.0
353,Plantain (bunch),NaN,2603.0,36442.0,14.0
392,Tomato Paste (tin),NaN,377.0,4901.0,13.0
444,Detergent 1kg,NaN,1804.0,5412.0,3.0


In [14]:
df["Quantity"] = df["Quantity"].fillna(df["Quantity_Calculated"])
df["Quantity"].isnull().sum()

0

In [15]:
# Missing Unit Price
df[df["UnitPrice_FCFA"].isnull()]

,TransactionID,Date,Product,Category,Quantity,UnitPrice_FCFA,TotalSale_FCFA,PaymentMethod,SalesRep,CustomerType,Quantity_Calculated
37,NKW-0038,2026-04-06,Rice 50kg,Grains,9.0,NaN,334566.0,Orange Money,Achu N.,Walk-in,NaN
42,NKW-0043,2026-03-21,Beans (White),Grains,19.0,NaN,16758.0,Bank Transfer,Ernest M.,Walk-in,NaN
125,NKW-0126,2026-06-14,Rice 50kg,Grains,13.0,NaN,485563.0,Orange Money,Ernest M.,Wholesale,NaN
149,NKW-0150,2026-02-05,Bread (loaf),Bakery,3.0,NaN,1827.0,Orange Money,Achu N.,Walk-in,NaN
400,NKW-0401,2026-03-26,Matches (box),Household,3.0,NaN,294.0,Orange Money,Beatrice T.,Walk-in,NaN


In [16]:
df["CalculatedUnitPrice"] = (
    df["TotalSale_FCFA"] / df["Quantity"]
)

In [17]:
df[df["UnitPrice_FCFA"].isnull()][
    ["Product", "Quantity", "UnitPrice_FCFA", "TotalSale_FCFA", "CalculatedUnitPrice"]
]

,Product,Quantity,UnitPrice_FCFA,TotalSale_FCFA,CalculatedUnitPrice
37,Rice 50kg,9.0,NaN,334566.0,37174.0
42,Beans (White),19.0,NaN,16758.0,882.0
125,Rice 50kg,13.0,NaN,485563.0,37351.0
149,Bread (loaf),3.0,NaN,1827.0,609.0
400,Matches (box),3.0,NaN,294.0,98.0


##### The unit price has 5 missing values

In [18]:
df["UnitPrice_FCFA"] = df["UnitPrice_FCFA"].fillna(df["CalculatedUnitPrice"])
df["UnitPrice_FCFA"].isnull().sum()

0

In [19]:
# Checking
df["CalculatedTotal"] = (
    df["Quantity"] * df["UnitPrice_FCFA"]
)
df["SaleDifference"] = (
    df["TotalSale_FCFA"] - df["CalculatedTotal"]
)
df["SaleDifference"].abs().sum()

0.0

In [20]:
# Removing temporary questions
df.drop(
    columns = ["Quantity_Calculated", "CalculatedUnitPrice", "CalculatedTotal", "SaleDifference"],
    errors = "ignore",
    inplace = True
)

In [21]:
# Handle missing PaymentMethod

df["PaymentMethod"] = df["PaymentMethod"].fillna("Unknown")

### Missing Payment Methods
##### Ther were 10 transactions with missing payment methods. Because the original data does not provide enough information to determine the actual payment method, these values were labeled "Unknown" rather than guessing a payment method.

In [22]:
# Handle missing SalesRep
df["SalesRep"] = df["SalesRep"].fillna("Unknown")

### Missing Sales Representatives
##### Six transactions had no recorded SalesRep. Since the responsibe sales representative cannot be reliably inferred, these values were labeed "Unknwon" rather than assigning a representative arbitrarily.

In [23]:
# Converting the date
df["Date"] = pd.to_datetime(df["Date"])
df["Date"].dtype

dtype('<M8[ns]')

In [24]:
# Checking for impossible values
df["Quantity"].describe()

count    500.000000
mean      10.988000
std       10.220802
min        1.000000
25%        5.000000
50%       11.000000
75%       15.000000
max      200.000000
Name: Quantity, dtype: float64

In [25]:
# Checks
print((df["Quantity"] < 0).sum())
print((df["UnitPrice_FCFA"] < 0).sum())
print((df["TotalSale_FCFA"] < 0).sum())

0
0
0


##### From the above, the columns Quantity, UnitPrice and TotalSale have no values less then 0

In [26]:
# Checking for the final missing 
df.isnull().sum()

TransactionID     0
Date              0
Product           0
Category          0
Quantity          0
UnitPrice_FCFA    0
TotalSale_FCFA    0
PaymentMethod     0
SalesRep          0
CustomerType      0
dtype: int64

##### The results show that, there are no longer missing values in the data set.

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   TransactionID   500 non-null    object        
 1   Date            500 non-null    datetime64[ns]
 2   Product         500 non-null    object        
 3   Category        500 non-null    object        
 4   Quantity        500 non-null    float64       
 5   UnitPrice_FCFA  500 non-null    float64       
 6   TotalSale_FCFA  500 non-null    float64       
 7   PaymentMethod   500 non-null    object        
 8   SalesRep        500 non-null    object        
 9   CustomerType    500 non-null    object        
dtypes: datetime64[ns](1), float64(3), object(6)
memory usage: 39.2+ KB


##### The data type of date has been changes successfully.

In [28]:
# Counting the unique products
df["Product"].nunique()

20

In [29]:
df["Product"].unique()

array(['Rice 50kg', 'Onions 1kg', 'Salt 1kg', 'Bread (loaf)',
       'Beans (White)', 'Rice 25kg', 'Palm Oil 1L', 'Tomatoes 1kg',
       'Palm Oil 5L', 'Tomato Paste (tin)', 'Detergent 1kg',
       'Beans (Red)', 'Matches (box)', 'Cassava (bag)', 'Sugar 1kg',
       'Maggi Cubes (pack)', 'Vegetable Oil 5L', 'Plantain (bunch)',
       'Milk Powder 400g', 'Soap (bar)'], dtype=object)

In [30]:
# Creating the product catalog
df_sorted = df.sort_values("Date")
products = (
    df_sorted.drop_duplicates("Product", keep = "last")   # Keeps the details of only the last in the sorted list
    [["Product", "Category", "UnitPrice_FCFA"]].copy()
)

In [31]:
products = products.rename(columns = {
    "Product": "name",
    "Category": "category",
    "UnitPrice_FCFA": "price"
})
products

,name,category,price
33,Bread (loaf),Bakery,585.0
450,Matches (box),Household,103.0
374,Palm Oil 5L,Oils,6364.0
293,Beans (White),Grains,847.0
201,Plantain (bunch),Produce,2390.0
413,Beans (Red),Grains,938.0
434,Maggi Cubes (pack),Groceries,503.0
373,Tomato Paste (tin),Groceries,355.0
194,Soap (bar),Household,390.0
453,Onions 1kg,Produce,748.0


In [32]:
# Exporting products.json

In [33]:
# Creating the notebook
products.to_json( "products.json", orient = "records", indent = 4)

In [34]:
# Check in json
with open ("products.json", "r", encoding = "utf-8") as file:
    product_data = json.load(file)
product_data[:3]    

[{'name': 'Bread (loaf)', 'category': 'Bakery', 'price': 585.0},
 {'name': 'Matches (box)', 'category': 'Household', 'price': 103.0},
 {'name': 'Palm Oil 5L', 'category': 'Oils', 'price': 6364.0}]

In [35]:
# ensuring that the length and names of the products json file is same as that of unique products

assert len(product_data) == products["name"].nunique()
assert products["name"].is_unique

## Conclusion
##### The raw Nkwen Traders sales dataset was inspected and cleaned before being prepared for use by the website.

##### The cleaning process include;
##### - Checking the dataset structure and data type.
##### - Identifying and handling missing values.
##### - Standardizing inconsistent category names.
##### - Correcting the "Mobile Moeny" payment-method typo
##### - Checking for duplicate transactions
##### - Converting dates to a proper datetim format
##### - Validating the relationship between, quantity, unit price and total sale
##### - Creating a unique product prcv
##### - Exporting the catalog